# 웨이퍼 결함 분류 결과 시각화 생성

LSWMD 데이터와 `modeling/artifacts`의 완료 실험 결과를 읽어 데이터 및 모델 분석용 PNG를 생성한다. 모든 이미지는 650dpi로 저장한다. 기존 산출물 보호를 위해 같은 이름의 파일이 있으면 기본적으로 중단한다.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from PIL import Image

OUTPUT_DPI = 650
ALLOW_REGENERATE = False
CLASS_ORDER = [
    'Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc',
    'Random', 'Scratch', 'Near-full', 'none',
]
CLASS_KO = {
    'Center': '중앙', 'Donut': '도넛', 'Edge-Loc': '가장자리 국소',
    'Edge-Ring': '가장자리 링', 'Loc': '국소', 'Random': '무작위',
    'Scratch': '스크래치', 'Near-full': '전면', 'none': '정상',
}
COLORS = {
    'navy': '#17324D', 'blue': '#2F6BFF', 'cyan': '#3CBCC3',
    'orange': '#F4A261', 'red': '#E76F51', 'green': '#2A9D8F',
    'gray': '#7A8793', 'light': '#EEF3F8', 'ink': '#17212B',
}

cwd = Path.cwd().resolve()
if cwd.name == 'modeling':
    MODELING_ROOT = cwd
elif cwd.name == 'docs' and cwd.parent.name == 'modeling':
    MODELING_ROOT = cwd.parent
else:
    MODELING_ROOT = cwd / 'modeling'
PROJECT_ROOT = MODELING_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'LSWMD.pkl'
ARTIFACT_ROOT = MODELING_ROOT / 'artifacts'
SUITE_DIR = ARTIFACT_ROOT / 'suites' / '4048895abf6e'
IMAGE_DIR = MODELING_ROOT / 'docs' / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans'],
    'axes.unicode_minus': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 15,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

generated_paths: list[Path] = []


def save_figure(fig: plt.Figure, filename: str) -> Path:
    """그림을 고해상도 PNG로 저장하고 경로를 반환한다.

    Args:
        fig: 저장할 Matplotlib Figure.
        filename: `images` 폴더 아래의 PNG 파일명.

    Returns:
        저장한 이미지의 절대 경로.

    Raises:
        FileExistsError: 기존 파일 덮어쓰기가 허용되지 않은 경우.
    """
    path = IMAGE_DIR / filename
    if path.exists() and not ALLOW_REGENERATE:
        raise FileExistsError(
            f'기존 파일을 보호하기 위해 중단했습니다: {path}'
        )
    fig.savefig(
        path, dpi=OUTPUT_DPI, bbox_inches='tight',
        facecolor='white', metadata={'dpi': str(OUTPUT_DPI)},
    )
    plt.close(fig)
    generated_paths.append(path)
    return path


def add_box(
    ax: plt.Axes, xy: tuple[float, float], width: float, height: float,
    title: str, body: str, color: str,
) -> None:
    """좌표축에 설명 상자를 추가한다.

    Args:
        ax: 상자를 그릴 좌표축.
        xy: 상자 왼쪽 아래 좌표.
        width: 상자 너비.
        height: 상자 높이.
        title: 상자 제목.
        body: 상자 본문.
        color: 상자 테두리와 제목 색상.
    """
    box = FancyBboxPatch(
        xy, width, height, boxstyle='round,pad=0.018,rounding_size=0.025',
        linewidth=2, edgecolor=color, facecolor='white',
    )
    ax.add_patch(box)
    ax.text(
        xy[0] + width * 0.06, xy[1] + height * 0.80, title,
        fontsize=11.2, fontweight='bold', color=color, va='center',
    )
    ax.text(
        xy[0] + width * 0.06, xy[1] + height * 0.30, body,
        fontsize=8.8, color=COLORS['ink'], va='center', linespacing=1.4,
    )


## 1. 데이터와 실험 결과 로드

원본 811,457행 중 라벨이 존재하는 172,950행만 EDA에 사용한다. 실험 비교는 완료된 suite의 JSON/CSV 산출물만 사용한다.

In [ ]:
def unwrap_label(value: object) -> object:
    """중첩 배열 라벨을 단일 값으로 변환한다.

    Args:
        value: 원본 failureType 값.

    Returns:
        중첩을 제거한 라벨 또는 결측값.
    """
    while isinstance(value, (list, tuple, np.ndarray)):
        if np.size(value) == 0:
            return None
        value = np.asarray(value, dtype=object).reshape(-1)[0]
    return value


raw = pd.read_pickle(DATA_PATH)
labels = raw['failureType'].map(unwrap_label)
labeled_mask = labels.notna() & labels.ne('')
frame = raw.loc[labeled_mask, ['waferMap', 'lotName']].copy()
frame['failureType'] = labels.loc[labeled_mask].astype(str)
frame['source_index'] = np.flatnonzero(labeled_mask.to_numpy())
frame.reset_index(drop=True, inplace=True)
raw_rows = len(raw)
del raw, labels, labeled_mask

shape_values = frame['waferMap'].map(lambda value: np.asarray(value).shape)
frame['height'] = shape_values.map(lambda value: value[0])
frame['width'] = shape_values.map(lambda value: value[1])
frame['aspect_ratio'] = frame['width'] / frame['height']
class_counts = frame['failureType'].value_counts().reindex(CLASS_ORDER)

suite_status = json.loads((SUITE_DIR / 'status.json').read_text('utf-8'))
final_summary = json.loads(
    (SUITE_DIR / 'final_summary.json').read_text('utf-8')
)
records = []
for experiment_id in suite_status['completed_ids']:
    experiment_dir = ARTIFACT_ROOT / experiment_id
    config = json.loads(
        (experiment_dir / 'config.json').read_text('utf-8')
    )['config']
    metrics = json.loads(
        (experiment_dir / 'metrics.json').read_text('utf-8')
    )
    records.append({
        'experiment_id': experiment_id, **config,
        'validation_macro_f1': metrics['validation']['macro_f1'],
        'validation_worst_f1': metrics['validation']['worst_class_f1'],
        'validation_accuracy': metrics['validation']['accuracy'],
        'parameters': metrics['parameters'],
        'best_epoch': metrics['best_epoch'],
        'epochs_run': metrics['epochs_run'],
        'elapsed_seconds': metrics['elapsed_seconds'],
    })
experiments = pd.DataFrame(records).set_index('experiment_id')

BASELINE_ID = 'small_cnn_17488d1a4881'
FINAL_ID = 'residual_cnn_483f70e6ae86'
baseline_f1 = experiments.loc[BASELINE_ID, 'validation_macro_f1']
final_validation_f1 = experiments.loc[FINAL_ID, 'validation_macro_f1']
test_macro_mean = final_summary['macro_f1_mean']
test_macro_std = final_summary['macro_f1_std']
test_accuracy_mean = np.mean([row['accuracy'] for row in final_summary['runs']])

print(f'원본/라벨 데이터: {raw_rows:,} / {len(frame):,}')
print(f'검증 Macro-F1: {baseline_f1:.4f} -> {final_validation_f1:.4f}')
print(f'테스트 Macro-F1: {test_macro_mean:.4f} ± {test_macro_std:.4f}')


## 2. 프로젝트 요약과 이해관계자

표지용 KPI 카드, 이해관계자-페인포인트 연결도, 누수 방지형 실험 전략을 생성한다.

In [ ]:
# 설명형 발표 자료는 결과 문서의 텍스트로 작성한다.


## 3. EDA와 전처리

클래스 불균형, 대표 웨이퍼 맵, 원본 크기/종횡비, 리사이즈 방식 차이를 시각화한다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))
pie_colors = [COLORS['red'], COLORS['orange'], '#D9825B', '#C75C5C', COLORS['cyan'], '#6C8EBF', '#9B7EBD', '#D4A5A5', COLORS['blue']]
wedges, _ = ax.pie(class_counts.values, colors=pie_colors, startangle=90, wedgeprops={'linewidth': 1, 'edgecolor': 'white'})
legend_labels = [f'{CLASS_KO[name]} ({class_counts[name]:,}, {class_counts[name] / len(frame):.2%})' for name in CLASS_ORDER]
ax.legend(wedges, legend_labels, loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=False, fontsize=9)
ax.set_aspect('equal')
fig.tight_layout()
save_figure(fig, '03_class_distribution.png')
representatives = {}
for class_name in CLASS_ORDER:
    subset = frame.loc[frame['failureType'].eq(class_name)].head(250)
    ratios = subset['waferMap'].map(lambda value: np.mean(np.asarray(value) == 2))
    representatives[class_name] = subset.loc[(ratios - ratios.median()).abs().idxmin(), 'waferMap']
wafer_cmap = ListedColormap(['#17212B', '#F5F7FA', '#E63946'])
fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for ax, class_name in zip(axes.flat, CLASS_ORDER):
    wafer_map = np.asarray(representatives[class_name])
    ax.imshow(wafer_map, cmap=wafer_cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'{CLASS_KO[class_name]} ({class_name})', fontsize=10)
    ax.axis('off')
fig.tight_layout(rect=(0, 0.04, 1, 0.96))
save_figure(fig, '04_wafer_examples.png')
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.hist(frame['aspect_ratio'], bins=45, color=COLORS['blue'], alpha=0.85)
ax.axvline(1, color=COLORS['red'], linestyle='--', label='정사각형')
ax.set_xlabel('너비 / 높이')
ax.set_ylabel('샘플 수')
ax.legend()
fig.tight_layout()
save_figure(fig, '05_aspect_ratio_distribution.png')

shape_counts = frame.groupby(['height', 'width']).size().sort_values(ascending=False).head(15)
shape_labels = [f'{height}×{width}' for height, width in shape_counts.index]
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.barh(shape_labels[::-1], shape_counts.values[::-1], color=COLORS['cyan'])
ax.set_xlabel('샘플 수')
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
save_figure(fig, '05_common_map_sizes.png')

def resize_fixed(wafer_map: np.ndarray, size: int=64) -> np.ndarray:
    """종횡비를 무시하고 최근접 보간으로 리사이즈한다.

    Args:
        wafer_map: 원본 범주형 웨이퍼 맵.
        size: 출력 정사각형 한 변.

    Returns:
        리사이즈한 범주형 맵.
    """
    image = Image.fromarray(np.asarray(wafer_map, dtype=np.uint8))
    return np.asarray(image.resize((size, size), Image.Resampling.NEAREST))

def resize_pad(wafer_map: np.ndarray, size: int=64) -> np.ndarray:
    """종횡비를 보존해 리사이즈하고 중앙 패딩한다.

    Args:
        wafer_map: 원본 범주형 웨이퍼 맵.
        size: 출력 정사각형 한 변.

    Returns:
        리사이즈와 중앙 패딩을 적용한 범주형 맵.
    """
    source = np.asarray(wafer_map, dtype=np.uint8)
    height, width = source.shape
    scale = size / max(height, width)
    new_height = max(1, round(height * scale))
    new_width = max(1, round(width * scale))
    image = Image.fromarray(source).resize((new_width, new_height), Image.Resampling.NEAREST)
    padded = np.zeros((size, size), dtype=np.uint8)
    top = (size - new_height) // 2
    left = (size - new_width) // 2
    padded[top:top + new_height, left:left + new_width] = np.asarray(image)
    return padded
candidates = frame.loc[frame['failureType'].ne('none')].head(5000).copy()
distortion = np.abs(np.log(candidates['aspect_ratio']))
target_distortion = distortion.quantile(0.9)
sample_index = (distortion - target_distortion).abs().idxmin()
sample = np.asarray(candidates.loc[sample_index, 'waferMap'])
fixed = resize_fixed(sample)
padded = resize_pad(sample)
mask = (padded > 0).astype(np.uint8)
fig, axes = plt.subplots(1, 4, figsize=(13, 4))
panels = [(sample, f'원본\n{sample.shape[0]}×{sample.shape[1]}', wafer_cmap, 2), (fixed, 'Fixed resize\n64×64 · 형상 왜곡', wafer_cmap, 2), (padded, 'Resize + Pad\n64×64 · 비율 보존', wafer_cmap, 2), (mask, 'Pad + Mask의 mask 채널\n유효 영역=1', 'Greys', 1)]
for ax, (image, title, cmap, vmax) in zip(axes, panels):
    ax.imshow(image, cmap=cmap, vmin=0, vmax=vmax, interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig.tight_layout(rect=(0, 0, 1, 0.92))
save_figure(fig, '06_preprocessing_comparison.png')


## 4. 모델링 선택 근거

모델, 손실/증강, 전처리를 동일 validation Macro-F1 기준으로 비교한다.

In [ ]:
model_ids = ['small_cnn_17488d1a4881', 'spatial_cnn_eaa6f09b055c', 'residual_cnn_8123daeff8cd', 'hybrid_cnn_3b52190d3bbe']
model_names = ['SmallCNN', 'SpatialCNN', 'ResidualCNN', 'HybridCNN']
model_rows = experiments.loc[model_ids]
bar_colors = [COLORS['gray'], COLORS['cyan'], COLORS['green'], COLORS['orange']]
fig, ax = plt.subplots(figsize=(7, 5.2))
bars = ax.bar(model_names, model_rows['validation_macro_f1'], color=bar_colors)
ax.set_ylim(0.84, 0.89)
ax.set_ylabel('Validation Macro-F1')
ax.grid(axis='y', alpha=0.2)
for bar, value in zip(bars, model_rows['validation_macro_f1']):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.001, f'{value:.3f}', ha='center', fontsize=9)
fig.tight_layout()
save_figure(fig, '08_model_macro_f1.png')

fig, ax = plt.subplots(figsize=(7, 5.2))
ax.scatter(
    model_rows['parameters'] / 1000,
    model_rows['validation_macro_f1'],
    s=model_rows['elapsed_seconds'] / 4,
    c=bar_colors,
    alpha=0.8,
    edgecolor='white',
    linewidth=1.5,
)
for name, (_, row) in zip(model_names, model_rows.iterrows()):
    ax.annotate(name, (row['parameters'] / 1000, row['validation_macro_f1']), xytext=(5, 6), textcoords='offset points', fontsize=9)
ax.set_xlabel('파라미터 수 (천 개)')
ax.set_ylabel('Validation Macro-F1')
ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, '08_model_complexity.png')
recipe_ids = ['residual_cnn_e19d7dd38d6c', 'residual_cnn_345f6ea69a42', 'residual_cnn_8123daeff8cd', 'residual_cnn_ff67addb590f']
recipe_names = ['CE / 증강 없음', 'CE / 증강', 'Weighted CE / 증강 없음', 'Weighted CE / 증강']
recipe_rows = experiments.loc[recipe_ids]
fig, ax = plt.subplots(figsize=(10.5, 5.5))
recipe_colors = [COLORS['green'], COLORS['gray'], COLORS['gray'], COLORS['gray']]
bars = ax.barh(recipe_names[::-1], recipe_rows['validation_macro_f1'].values[::-1], color=recipe_colors[::-1])
ax.set_xlim(0.85, 0.89)
ax.set_xlabel('Validation Macro-F1')
ax.grid(axis='x', alpha=0.2)
for bar, value in zip(bars, recipe_rows['validation_macro_f1'].values[::-1]):
    ax.text(value + 0.0005, bar.get_y() + bar.get_height() / 2, f'{value:.3f}', va='center')
fig.tight_layout()
save_figure(fig, '09_recipe_comparison.png')
preprocess_ids = ['residual_cnn_e19d7dd38d6c', 'residual_cnn_483f70e6ae86', 'residual_cnn_4f3752a9111c']
preprocess_names = ['Fixed resize', 'Resize + Pad', 'Resize + Pad + Mask']
preprocess_rows = experiments.loc[preprocess_ids]
fig, ax = plt.subplots(figsize=(9.5, 5.5))
preprocess_colors = [COLORS['gray'], COLORS['green'], COLORS['orange']]
bars = ax.bar(preprocess_names, preprocess_rows['validation_macro_f1'], color=preprocess_colors)
ax.set_ylim(0.84, 0.895)
ax.set_ylabel('Validation Macro-F1')
ax.grid(axis='y', alpha=0.2)
fixed_value = preprocess_rows.iloc[0]['validation_macro_f1']
for bar, value in zip(bars, preprocess_rows['validation_macro_f1']):
    delta = (value - fixed_value) * 100
    label = f'{value:.3f}' if abs(delta) < 1e-09 else f'{value:.3f}\n({delta:+.2f}%p)'
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.001, label, ha='center', fontsize=10)
fig.tight_layout()
save_figure(fig, '10_preprocessing_performance.png')


## 5. 평가와 비즈니스 해석

학습 곡선, seed 안정성, 클래스별 성능, 혼동행렬, 운영 액션 매트릭스를 생성한다.

In [ ]:
history = pd.read_csv(ARTIFACT_ROOT / FINAL_ID / 'history.csv')
best_epoch = int(experiments.loc[FINAL_ID, 'best_epoch'])
fig, ax = plt.subplots(figsize=(7, 5.2))
ax.plot(history['epoch'], history['validation_macro_f1'], color=COLORS['blue'], linewidth=2, label='Validation Macro-F1')
ax.plot(history['epoch'], history['best_validation_macro_f1'], color=COLORS['green'], linestyle='--', label='Best so far')
ax.axvline(best_epoch, color=COLORS['red'], linestyle=':', label=f'최고 epoch {best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Macro-F1')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, '11_validation_macro_f1.png')

fig, ax = plt.subplots(figsize=(7, 5.2))
ax.plot(history['epoch'], history['train_loss'], color=COLORS['gray'], label='Train loss')
ax.plot(history['epoch'], history['validation_loss'], color=COLORS['orange'], label='Validation loss')
ax.axvline(best_epoch, color=COLORS['red'], linestyle=':')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, '11_loss_curve.png')
final_ids = [row['experiment_id'] for row in final_summary['runs']]
seeds = [row['seed'] for row in final_summary['runs']]
validation_scores = experiments.loc[final_ids, 'validation_macro_f1'].to_numpy()
test_scores = np.array([row['macro_f1'] for row in final_summary['runs']])
x = np.arange(len(seeds))
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.plot(x, validation_scores, 'o-', color=COLORS['blue'], linewidth=2, markersize=8, label='Validation')
ax.plot(x, test_scores, 'o-', color=COLORS['green'], linewidth=2, markersize=8, label='Test')
ax.axhline(test_scores.mean(), color=COLORS['green'], linestyle='--', alpha=0.7, label=f'Test 평균 {test_scores.mean():.3f}')
ax.fill_between(x, test_scores.mean() - test_scores.std(ddof=1), test_scores.mean() + test_scores.std(ddof=1), color=COLORS['green'], alpha=0.12)
ax.set_xticks(x, [str(seed) for seed in seeds])
ax.set_xlabel('Random seed')
ax.set_ylabel('Macro-F1')
ax.set_ylim(0.85, 0.9)
ax.legend()
ax.grid(alpha=0.2)
for index, value in enumerate(test_scores):
    ax.text(index, value - 0.004, f'{value:.3f}', ha='center', color=COLORS['green'])
fig.tight_layout()
save_figure(fig, '12_seed_stability.png')
class_f1 = pd.DataFrame({row['seed']: {name: values['f1'] for name, values in row['per_class'].items()} for row in final_summary['runs']}).reindex(CLASS_ORDER)
class_recall = pd.DataFrame({row['seed']: {name: values['recall'] for name, values in row['per_class'].items()} for row in final_summary['runs']}).reindex(CLASS_ORDER)
support = pd.Series({name: final_summary['runs'][0]['per_class'][name]['support'] for name in CLASS_ORDER})
means = class_f1.mean(axis=1)
stds = class_f1.std(axis=1, ddof=1)
order = means.sort_values().index
performance_colors = [COLORS['red'] if means[name] < 0.8 else COLORS['green'] for name in order]
fig, ax = plt.subplots(figsize=(7.5, 6.2))
ax.barh([CLASS_KO[name] for name in order], means.loc[order], xerr=stds.loc[order], color=performance_colors, alpha=0.9, capsize=3)
ax.set_xlim(0.65, 1.01)
ax.set_xlabel('Test F1 (3-seed 평균 ± 표준편차)')
ax.grid(axis='x', alpha=0.2)
for index, name in enumerate(order):
    ax.text(means[name] + 0.008, index, f'{means[name]:.3f}', va='center', fontsize=9)
fig.tight_layout()
save_figure(fig, '13_class_f1.png')

fig, ax = plt.subplots(figsize=(7.5, 6.2))
ax.barh([CLASS_KO[name] for name in order], support.loc[order], color=COLORS['blue'], alpha=0.85)
ax.set_xscale('log')
ax.set_xlabel('클래스별 test support (로그 축)')
ax.grid(axis='x', alpha=0.2, which='both')
for index, name in enumerate(order):
    ax.text(support[name] * 1.08, index, f'{support[name]:,}', va='center', fontsize=9)
fig.tight_layout()
save_figure(fig, '13_class_support.png')
confusion = pd.DataFrame(0, index=CLASS_ORDER, columns=CLASS_ORDER, dtype=int)
for experiment_id in final_ids:
    predictions = pd.read_csv(ARTIFACT_ROOT / experiment_id / 'test_predictions.csv', usecols=['true_label', 'predicted_label'])
    confusion += pd.crosstab(predictions['true_label'], predictions['predicted_label']).reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)
normalized = confusion.div(confusion.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(normalized, cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(9), [CLASS_KO[name] for name in CLASS_ORDER], rotation=35, ha='right')
ax.set_yticks(range(9), [CLASS_KO[name] for name in CLASS_ORDER])
ax.set_xlabel('예측 클래스')
ax.set_ylabel('실제 클래스')
for row in range(9):
    for column in range(9):
        value = normalized.iloc[row, column]
        if value >= 0.5 or row == column:
            ax.text(column, row, f'{value:.1f}', ha='center', va='center', fontsize=8, color='white' if value > 55 else COLORS['ink'])
colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colorbar.set_label('비율 (%)')
fig.tight_layout()
save_figure(fig, '14_confusion_matrix.png')


## 6. 산출물 검증

생성 파일 수, 픽셀 크기, DPI 메타데이터를 확인한다. PNG의 물리 해상도 환산 오차를 고려해 650dpi로 저장했으므로 600dpi 이상이어야 한다.

In [ ]:
checks = []
for path in generated_paths:
    with Image.open(path) as image:
        dpi = image.info.get('dpi', (0, 0))
        checks.append({
            '파일': path.name, '너비(px)': image.width, '높이(px)': image.height,
            'DPI-X': round(dpi[0], 2), 'DPI-Y': round(dpi[1], 2),
            '600dpi 이상': min(dpi) >= 600,
        })
checks_frame = pd.DataFrame(checks)
if len(checks_frame) != 15 or not checks_frame['600dpi 이상'].all():
    raise RuntimeError('이미지 수 또는 DPI 검증에 실패했습니다.')
checks_frame
